---
image: example.gif
execute: 
  enabled: true
---

# Visualising a more complex status alongside entity icons - gas station with individual fuel tank level

In [ ]:
from vidigi.utils import EventPosition, create_event_position_df
from vidigi.prep import reshape_for_animations, generate_animation_df
from vidigi.animation import generate_animation, animate_activity_log
import pandas as pd
import os
import random
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.graph_objects as go
import plotly.express as px
pio.renderers.default = "notebook"

In [ ]:
#| echo: false
#| output: asis
# Path to the external Python script
file_path = "simpy_gas_stations.py"

# Read the file content
if os.path.exists(file_path):
    with open(file_path, "r") as f:
        code_content = f.read()
else:
    code_content = "File not found."
with open(file_path, "r") as f:
    code_content = f.read()

# Print the Quarto `{details}` block for collapsible output
print(f"""
:::{{.callout-note collapse="true"}}
### View Imported Code, which has had logging steps added at the appropriate points in the 'model' class

```python
{code_content}
```

:::

""")

In [ ]:
# Define positions for animation
event_positions = create_event_position_df([
    EventPosition(event='arrival', x=0, y=350, label="Entrance"),
    EventPosition(event='pump_queue_wait_begins', x=400, y=350, label="Queue"),
    EventPosition(event='payment_begins', x=340, y=175, resource='num_pumps',
                  label="Pumping Gas"),
    EventPosition(event='pumping_begins', x=340, y=175, resource='num_pumps',
                  label="Pumping Gas"),
    EventPosition(event='calling_truck', x=140, y=50,
                  label="Calling Truck"),
        EventPosition(event='refuelling', x=340, y=50,
                  label="Truck Filling Tank"),
    EventPosition(event='depart', x=250, y=50, label="Exit")
])

class Params:
    def __init__(self):
        self.num_pumps = 2

icon_list = [ "🚗", "🚙", "🚓",
            "🚗", "🚙", "🏍️", "🏍️",
            "🚗", "🚙", "🚑",
            "🚗", "🚙", "🛻",
            "🚗", "🚙", "🚛",
            "🚗", "🚙", "🚕",
            "🚗", "🚙", "🚒",
            "🚗", "🚙", "🚑"]

random.shuffle(icon_list)

In [ ]:
event_log_df = pd.read_csv("gas_station_log.csv")

In [ ]:
STEP_SNAPSHOT_MAX = 6
LIMIT_DURATION = 60*60*3
WRAP_QUEUES_AT = 3

In [ ]:
full_entity_df = reshape_for_animations(
    event_log=event_log_df,
    every_x_time_units=5,
    step_snapshot_max=STEP_SNAPSHOT_MAX,
    limit_duration=LIMIT_DURATION,
    debug_mode=True
    )

full_entity_df_plus_pos = generate_animation_df(
    full_entity_df=full_entity_df,
    event_position_df=event_positions,
    wrap_queues_at=WRAP_QUEUES_AT,
    step_snapshot_max=STEP_SNAPSHOT_MAX,
    gap_between_entities=150,
    gap_between_resources=180,
    gap_between_queue_rows=150,
    # gap_between_resource_rows=60,
    debug_mode=True,
    custom_entity_icon_list=icon_list
    )


In [ ]:
def build_fuel_bar(value, max_value=50, length=10):
    """Create an ASCII bar to show fuel level."""
    try:
        if value is None or (isinstance(value, float) and (value != value)):  # check for None or NaN
            proportion = 0
        else:
            proportion = min(max(value / max_value, 0), 1)
    except Exception:
        proportion = 0  # fallback

    filled = int(proportion * length)
    empty = length - filled
    filled_icon = "█"
    empty_icon = "░"
    return "[" + filled_icon * filled + empty_icon * empty + "]"


In [ ]:
def custom_icon_rules(row):
    icon = row.get("icon", "")
    entity_id = row.get("entity_id", "")
    event = row.get("event", "")
    fuel_level_start = row.get("fuel_level_start", None)  # Only for cars

    if "more" not in str(icon):
        if isinstance(entity_id, str):
            if "Truck" in entity_id:
                return "🚚 Truck is refilling the tank..."
            elif "Call" in entity_id:
                return "☎️ Calling Truck!"
            elif "Car" in entity_id:
                bar = ""
                if (event == "arrival" or event == "pump_queue_wait_begins") and fuel_level_start is not None:
                    bar = " " + build_fuel_bar(fuel_level_start)
                    return icon + "<br>" + bar + "<br><br>"
                elif event == "payment_begins" and fuel_level_start is not None:
                    bar = " " + build_fuel_bar(fuel_level_start)
                    return icon+ "<br>" + bar + "<br> Paying"
                elif event == "pumping_begins" and fuel_level_start is not None:
                    arrival_time = row["time"]
                    elapsed = max(float(row["snapshot_time"]) - float(arrival_time), 0)
                    current_fuel = min(fuel_level_start + elapsed * 1, 50)
                    bar = " " + build_fuel_bar(current_fuel)
                    return icon+ "<br>" + bar + "<br> Pumping"
                elif event == "departure" or event == "pumping_ends":
                    bar = " " + build_fuel_bar(50)  # Car is full when it leaves
                    return icon+ "<br>" + bar + "<br> <br>"


            else:
                return icon
    return icon


full_entity_df_plus_pos = full_entity_df_plus_pos.assign(
            icon=full_entity_df_plus_pos.apply(custom_icon_rules, axis=1)
            )

In [ ]:
fig = generate_animation(
        full_entity_df_plus_pos=full_entity_df_plus_pos.sort_values(['entity_id', 'snapshot_time']),
        event_position_df= event_positions,
        scenario=Params(),
        simulation_time_unit="seconds",
        plotly_height=900,
        plotly_width=1200,
        override_x_max=500,
        override_y_max=750,
        entity_icon_size=30,
        gap_between_resources=180,
        display_stage_labels=False,
        # resource_opacity=1,
        resource_opacity=0,
        setup_mode=False,
        # custom_resource_icon="⛽",
        resource_icon_size=40,
        add_background_image="https://raw.githubusercontent.com/hsma-tools/vidigi/refs/heads/main/examples/example_15_gas_station_refuelling/gas_station.png",
        background_image_opacity=1, # New parameter in 1.1.0
        overflow_text_color="white", # New parameter in 1.1.0
        start_time="09:00:00",
        time_display_units="%H:%M:%S",
        debug_mode=True,
        frame_duration=100,
        frame_transition_duration=100
    )

fig

In [ ]:
fuel_level_change_df = event_log_df[(event_log_df["event_type"]=="fuel_level_change") &
                                    (event_log_df["time"] % 5 == 0) &
                                    (event_log_df["time"] < LIMIT_DURATION)]

px.bar(fuel_level_change_df, x="entity_id", y="value", animation_frame="time", range_y=[0,400])

## Explore incorporating the fuel level bar plot as an additional synchronised plot

In [ ]:
## Same as before, but increase the height to give space for some of it to be taken up by the bar plot later

fig = generate_animation(
        full_entity_df_plus_pos=full_entity_df_plus_pos.sort_values(['entity_id', 'snapshot_time']),
        event_position_df= event_positions,
        scenario=Params(),
        simulation_time_unit="seconds",
        plotly_height=1000,
        plotly_width=1200,
        override_x_max=500,
        override_y_max=750,
        entity_icon_size=30,
        gap_between_resources=180,
        display_stage_labels=False,
        # resource_opacity=1,
        resource_opacity=0,
        setup_mode=False,
        # custom_resource_icon="⛽",
        resource_icon_size=40,
        add_background_image="https://raw.githubusercontent.com/hsma-tools/vidigi/refs/heads/main/examples/example_15_gas_station_refuelling/gas_station.png",
        background_image_opacity=1, # New parameter in 1.1.0
        overflow_text_color="white", # New parameter in 1.1.0
        start_time="09:00:00",
        time_display_units="%H:%M:%S",
        debug_mode=True,
        frame_duration=100,
        frame_transition_duration=100
    )

In [ ]:
# Set up the desired subplot layout
ROWS = 2

sp = make_subplots(
    rows=ROWS,
    cols=1,
    row_heights=[0.75, 0.25],
    vertical_spacing=0.05,
    subplot_titles=(
        "", # Original Animation
        "Station Tank Fuel Level", # Fuel Tank Level
        )
    )

# Overwrite the domain of our original x and y axis with domain from the new axis
fig.layout['xaxis']['domain'] = sp.layout['xaxis']['domain']
fig.layout['yaxis']['domain'] = sp.layout['yaxis']['domain']

for i in range(2, ROWS+1):

    # Add in the attributes for the secondary axis from our subplot
    fig.layout[f'xaxis{i}'] = sp.layout[f'xaxis{i}']
    fig.layout[f'yaxis{i}'] = sp.layout[f'yaxis{i}']

fig._grid_ref = sp._grid_ref

In [ ]:
# First, extract the trace containing the resource icons
# icon_trace = fig.data[1]

# Now keep our figure data as just the initial trace.
fig.data = (fig.data[0],)

# 1. RESOURCE ICONS TRACE
# Readd the resource icons trace in a consistent manner
# Confusingly, when we start messing with the naimation frames, we lose the resource icon trace
# even though it appeared fine until this point - so we have to handle it here
# fig.add_trace(icon_trace)

# 2. BAR PLOT ON SECONDARY AXIS (animated barplot in subplot)
# Initialize with a single point and assign it to subplot axes (x2/y2)

# Get unique time points
time_points = fuel_level_change_df["time"].unique()

# Initial frame (first time point)
initial_time = time_points[0]
initial_df = fuel_level_change_df[fuel_level_change_df["time"] == initial_time]

# Create the initial bar trace

fig.add_trace(go.Bar(
    x=[fuel_level_change_df["entity_id"].values[0]],
    y=[fuel_level_change_df["value"].values[0]],
    showlegend=False
    # We place it in our new subplot using the following line
), row=2, col=1)

In [ ]:
# # Now ensure we tell it which traces we are animating
# # (as per https://chart-studio.plotly.com/~empet/15243/animating-traces-in-subplotsbr/#/)
for i, frame in enumerate(fig.frames):
    # Your original frame.data
    # This will be a tuple
    # We'll ensure we only take the first entry
    # original_data = (frame.data[0], )

    original_data = frame.data

    # The new data you want to add for this specific frame
    new_data = (
        # 0: resource icons
        # icon_trace,

        go.Bar(
            x= [fuel_level_change_df.sort_values('time')['entity_id'].values[i]],
            y= [fuel_level_change_df.sort_values('time')['value'].values[i]]
        ) ,  # This needs to be a tuple even if we're only adding a single additional trace, hence the comma
    )

    frame.data = original_data + new_data


In [ ]:
fig